# 최고점 0.520406 · 저장 모델 재현 및 제출 CSV

**런타임 → 런타임 유형 변경 → GPU**, 이어서 **모두 실행**을 선택하세요.
Google 인증 요청은 이 패키지가 있는 기존 팀 Drive에 접근하기 위한 것입니다.

이 노트북은 **새 학습을 하지 않습니다**. 원자료를 저장된 전처리기에 통과시켜
① 기존 OOF 점수를 재현하고 ② 2,546행 제출 CSV를 다시 만듭니다.
모델: fold별 mRMR+global+raw12 XGBoost와 TF-IDF C10 로지스틱,
내부 검증에서 이미 학습된 C10 클래스 보정, 동일 비율 기하평균, 입력 기반 H1 fallback.
제출 예측은 다섯 fold의 최종 확률을 산술평균한 뒤 클래스를 선택합니다.

- 내부 기준: **6,201행 · 26개 클래스 · pooled OOF Macro-F1 0.5204059148667912**.
- 이는 반복 탐색한 seed43 개발 점수이며, 독립 검증이나 제출 서버 점수가 아닙니다.
- test는 추론에만 사용하며 전처리·모델·보정값을 다시 맞추지 않습니다.
- 같은 Windows GPU에서는 확률 오차 1e-12 이하를 검사합니다. 다른 Colab GPU의
  수치 차이는 명시적으로 1e-6까지 허용하되 **OOF 예측 클래스·점수와 최종 CSV 해시는 같아야** 합니다.
- Mac CPU에서 저장 모델 OOF를 재현한 결과가 기준입니다. **실제 Colab GPU 실행은 아직 검증되지 않았습니다.**

이것은 저장 모델 기반 재현본이며 처음부터 모델을 재학습하는 노트북은 아닙니다.
새 탐색은 중단된 상태입니다. 대회 사이트로 제출하는 기능도 없습니다.


In [ ]:
#@title 1. Drive 인증과 경로 설정
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import platform
import subprocess
import sys
import urllib.request
import uuid
import zipfile
import google.auth
from google.colab import auth, files
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google_auth_httplib2 import AuthorizedHttp
import httplib2
auth.authenticate_user()
credentials, _ = google.auth.default()
SERVICE = build("drive", "v3", http=AuthorizedHttp(credentials, http=httplib2.Http(timeout=120)), cache_discovery=False)
ROOT = Path("/content/cbj_best05204")
OUTPUT = Path("/content/cbj_best05204_outputs") / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
print("결과 저장 위치:", OUTPUT)


In [ ]:
#@title 2. 동결 패키지 다운로드와 무결성 검사
BUNDLE_ID = '1gm6rPuAAowgwYC5BHZ2oRaZ_3dZSd2Q6'
BUNDLE_SHA256 = '6dd0686bce32f3709048df7d42a6ef24fd8ef149697aa14b2f57095ddbeb62f4'
def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def extract_verified_bundle(archive_path, destination, expected_sha256):
    """Verify the entire archive and all target paths before writing any member."""
    archive_path, destination = Path(archive_path), Path(destination).resolve()
    if digest(archive_path) != expected_sha256:
        raise ValueError("Frozen bundle SHA-256 mismatch")
    with zipfile.ZipFile(archive_path) as archive:
        entries, seen, total = [], set(), 0
        for item in archive.infolist():
            name = PurePosixPath(item.filename)
            target = (destination / item.filename).resolve()
            if (name.is_absolute() or ".." in name.parts or "\\" in item.filename
                    or ":" in item.filename or not target.is_relative_to(destination)
                    or target == destination or target in seen
                    or (item.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError("Unsafe or duplicate ZIP member")
            seen.add(target)
            total += item.file_size
            if total > 2 * 1024**3 or item.file_size > 512 * 1024**2:
                raise ValueError("Unexpected bundle size")
            if not item.is_dir():
                entries.append((item, target))
        for item, target in entries:
            if target.exists() and (not target.is_file() or
                                   target.read_bytes() != archive.read(item)):
                raise ValueError("Existing frozen file differs: " + item.filename)
        if archive.testzip() is not None:
            raise ValueError("ZIP integrity failure")
        for item, target in entries:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                target.write_bytes(archive.read(item))

archive_path = OUTPUT / "cbj_best05204_frozen.zip"
with archive_path.open("xb") as stream:
    download = MediaIoBaseDownload(stream, SERVICE.files().get_media(fileId=BUNDLE_ID, supportsAllDrives=True))
    done = False
    while not done:
        progress, done = download.next_chunk(num_retries=4)
        if progress is not None:
            print(f"다운로드 {progress.progress():.0%}")
extract_verified_bundle(archive_path, ROOT, BUNDLE_SHA256)
print("패키지 SHA-256 및 안전한 압축 해제 검사 통과")


In [ ]:
#@title 3. Windows와 같은 Python 3.12.12 및 전체 패키지 버전 설치
if platform.machine() != "x86_64" or sys.platform != "linux":
    raise RuntimeError("This replay expects Linux x86_64.")
TOOLS=ROOT/"tools"
TOOLS.mkdir(exist_ok=True)
with urllib.request.urlopen("https://pypi.org/pypi/uv/0.9.18/json",timeout=30) as response:
    metadata=json.load(response)
wheels=[f for f in metadata["urls"] if f["filename"].endswith(".whl") and "manylinux_2_17_x86_64" in f["filename"]]
if len(wheels)!=1:
    raise RuntimeError("Ambiguous environment bootstrap wheel")
spec=wheels[0]
with urllib.request.urlopen(spec["url"],timeout=60) as response:
    content=response.read()
if hashlib.sha256(content).hexdigest()!=spec["digests"]["sha256"]:
    raise RuntimeError("Environment tool hash mismatch")
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    members=[n for n in archive.namelist() if n.endswith("/uv")]
    if len(members)!=1:
        raise RuntimeError("Unexpected wheel contents")
    (TOOLS/"uv").write_bytes(archive.read(members[0]))
(TOOLS/"uv").chmod(0o755)
ENV={**os.environ,"UV_PYTHON_INSTALL_DIR":str(ROOT/"python"),"UV_CACHE_DIR":str(ROOT/"uv-cache"),
     "UV_PYTHON_BIN_DIR":str(TOOLS),"UV_LINK_MODE":"copy"}
subprocess.run([str(TOOLS/"uv"),"python","install","3.12.12"],env=ENV,check=True)
if not (ROOT/".venv/bin/python").exists():
    subprocess.run([str(TOOLS/"uv"),"venv","--python","3.12.12",str(ROOT/".venv")],env=ENV,check=True)
PYTHON=ROOT/".venv/bin/python"
subprocess.run([str(TOOLS/"uv"),"pip","sync","--python",str(PYTHON),str(ROOT/"requirements.lock.txt")],env=ENV,check=True)
print("Isolated replay environment is ready")


In [ ]:
#@title 4. 원자료에서 저장 모델 OOF 재추론 · 새 학습 없음
ENV.update({"OMP_NUM_THREADS": "4", "OPENBLAS_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"})
REPLAY_TOLERANCE = 1e-6
def run_command(command):
    with (OUTPUT / "execution.log").open("a", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=ROOT, env=ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError("재현 검사 실패: execution.log를 확인하세요. 자동 재시도/설정 변경은 하지 않습니다.")
RUNNER = ROOT / "best05204_infer.py"
run_command([str(PYTHON), str(RUNNER), "--bundle-root", str(ROOT), "--mode", "audit",
             "--out", str(OUTPUT / "oof_audit"), "--device", "cuda:0",
             "--tolerance", str(REPLAY_TOLERANCE)])


In [ ]:
#@title 5. 내부점수·fold 분산·과적합 위험 다시 확인
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, log_loss

with np.load(OUTPUT / "oof_audit/audit_replay_oof.npz", allow_pickle=False) as z:
    y, folds, prob = z["y"], z["folds"], z["prob"]
fold_scores = [f1_score(y[folds == fold], prob[folds == fold].argmax(1),
                        labels=np.arange(26), average="macro", zero_division=0)
               for fold in sorted(np.unique(folds))]
summary = pd.DataFrame({
    "metric": ["pooled_macro_f1", "fold_mean", "fold_sd", "fold_min", "fold_max", "logloss"],
    "value": [f1_score(y, prob.argmax(1), labels=np.arange(26), average="macro", zero_division=0),
              np.mean(fold_scores), np.std(fold_scores), np.min(fold_scores),
              np.max(fold_scores), log_loss(y, prob, labels=np.arange(26))],
})
display(summary)
print("fold Macro-F1:", [round(value, 6) for value in fold_scores])

c10_reviews = [json.loads((ROOT / f"folds/fold_{fold}/c10/review.json").read_text())
               for fold in range(5)]
train_mean = np.mean([row["train_metrics"]["macro_f1"] for row in c10_reviews])
valid_mean = np.mean([row["valid_metrics"]["macro_f1"] for row in c10_reviews])
print(f"C10 train Macro-F1 mean={train_mean:.6f}")
print(f"C10 outer-valid Macro-F1 mean={valid_mean:.6f}")
print(f"C10 mean train-valid gap={train_mean-valid_mean:.6f}")
print("판정: C10 구성요소에는 큰 과적합 압력이 있습니다.")
print("최종 앙상블의 in-sample 확률은 없으므로 최종 모델 train-valid gap은 미측정입니다.")
print("seed43 적응 개발 최고점이며 seed44 독립 확인은 없습니다.")


In [ ]:
#@title 6. 같은 저장 모델로 제출 CSV 재생성
SUBMISSION = OUTPUT / "submission_best05204.csv"
run_command([str(PYTHON), str(RUNNER), "--bundle-root", str(ROOT), "--mode", "infer",
             "--out", str(SUBMISSION), "--device", "cuda:0",
             "--tolerance", str(REPLAY_TOLERANCE)])


In [ ]:
#@title 7. 검증된 기준 CSV와 완전 일치 확인 후 다운로드
EXPECTED_CSV_SHA256 = '84694efbf9beedbfc34f9d40c7c395278321714b97d2805de8ca4d1371aeff14'
if digest(SUBMISSION) != EXPECTED_CSV_SHA256:
    raise RuntimeError("검증된 기준 제출 CSV와 다릅니다. 제출하지 말고 재현 결과를 확인하세요.")
import pandas as pd
submission = pd.read_csv(SUBMISSION, dtype=str)
sample = pd.read_csv(ROOT / "data/sample_submission.csv", dtype=str)
assert submission.columns.tolist() == ["ID", "SUBCLASS"]
assert len(submission) == 2546 and submission.ID.is_unique
assert submission.ID.tolist() == sample.ID.tolist()
assert not submission.isna().any().any()
print("2,546행 · ID 순서 · 결측 없음 · 기준 CSV SHA-256 일치")
print("새 학습 0회 / 대회 사이트 제출 없음")
display(submission.head())
files.download(str(SUBMISSION))


## 사용 범위와 실패 시 확인

결과의 `execution.log`와 감사 영수증을 함께 보관하세요.
Google 권한 오류는 이 패키지에 접근 가능한 계정으로 인증하고 다시 시도해야 합니다.
GPU가 없거나 OOF/CSV가 기준과 다르면 자동으로 다른 모델·CPU·보정값으로 바꾸지 않습니다.
일부 셀만 실행한 결과를 전체 재현 성공으로 취급하지 마세요.
원형 학습 코드와 설정은 패키지의 출처 기록으로 보존되어 있으나 이 노트북에서는 실행하지 않습니다.
